In [1]:
from nsnet2_denoiser import NSnet2Enhancer

from torch_stoi import NegSTOILoss
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility
from torchaudio.transforms import Resample

import torch
import torchaudio
import numpy as np

import os

In [2]:
SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)

gen = torch.Generator()
gen.manual_seed(SEED)

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [17]:
SR = 48_000

NOISE_PATH = "data/DS_10283_2791/clean_testset_wav"
CLEAN_PATH = "data/DS_10283_2791/clean_testset_wav"

noise_paths = [os.path.join(NOISE_PATH, x) for x in os.listdir(NOISE_PATH)]
clean_paths = [os.path.join(CLEAN_PATH, x) for x in os.listdir(CLEAN_PATH)]

test_data = list(zip(noise_paths, clean_paths))

BATCH_SIZE = 32
DEVICE = "cuda:0"

In [18]:
srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
stoi = NegSTOILoss(SR, use_vad=False, do_resample=False).to(DEVICE)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to(DEVICE)

In [19]:
enhancer = NSnet2Enhancer(fs=SR)

In [35]:
from tqdm import tqdm

def get_metrics(data, device="cpu"):
    nisqa_scores = []
    pesq_scores = []
    stoi_scores = []
    srmr_scores = []
    with torch.no_grad():
        for input_path, target_path in tqdm(data):
            
            signal, signal_sr = torchaudio.load(input_path)
            target, target_sr = torchaudio.load(target_path)

            signal = signal.to(device)
            target = target.to(device)
            
            # print(signal.shape)
            output = torch.from_numpy(enhancer(signal[0].cpu(), signal_sr)).unsqueeze(0).to(device)

            min_l = min(output.shape[-1], target.shape[-1])

            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

            stoi_score = stoi(output[..., :min_l], target[..., :min_l])
            srmr_score = srmr(output.detach().cpu())
            
            resampler = Resample(SR, 16_000)
            output = resampler(output.cpu()).cuda()
            target = resampler(target.cpu()).cuda()
            
            min_l = min(output.shape[-1], target.shape[-1])

            srmr_score = srmr(output.detach().cpu())

            try:
                pesq_score = pesq(output[..., :min_l], target[..., :min_l])
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            nisqa_scores.append(nisqa_score[0])
            srmr_scores.append(srmr_score)
            stoi_scores.append(stoi_score.cpu())
            pesq_scores.append(pesq_score.cpu())

    nisqa_scores = torch.vstack(nisqa_scores).mean(dim=0)
    stoi_scores = torch.vstack(stoi_scores).mean(dim=0)
    srmr_scores = torch.vstack(srmr_scores).mean(dim=0)
    pesq_scores = torch.vstack(pesq_scores).mean(dim=0)

    result = {"nisqa": nisqa_scores, "stoi": stoi_scores, "srmr": srmr_scores, "pesq": pesq_scores}
        
    return result

In [36]:
metrics = get_metrics(test_data, device=DEVICE)

 14%|█▍        | 115/824 [01:51<11:26,  1.03it/s]


KeyboardInterrupt: 

In [ ]:
print("NISQA score (MOS, NOI, DISC, COL, LOUD):", metrics["nisqa"])
print(f"STOI score: {metrics["stoi"]}")
print(f"SRMR score: {metrics["srmr"]}")
print(f"PESQ-WB score: {metrics["pesq"]}")